# Employee Attrition Prediction — IBM HR Analytics
### Internship Project — Week 2

**Goal:** Predict whether an employee is likely to leave the company, understand *why*, and
turn that into HR-actionable recommendations.

**Design principles used in this notebook (overcoming common pitfalls in a typical first pass):**
- All preprocessing (scaling, encoding) is fit **only on the training set** and applied to test
  data via a single `sklearn.Pipeline` — this prevents data leakage, which happens when you
  scale/encode on the full dataset *before* splitting.
- Model comparison uses **stratified 5-fold cross-validation** on the training set, not just a
  single train/test split — a single split is noisy on a dataset this small (1,470 rows).
- Class imbalance is handled with `class_weight='balanced'` consistently across the whole
  notebook — there is exactly **one** modeling pass, not a second contradictory SMOTE pass that
  silently overwrites earlier results.
- The decision threshold (0.5 by default) is explicitly revisited with a business lens: in
  attrition prediction, **missing a leaver (false negative) is usually costlier than a false
  alarm (false positive)**, so we evaluate Recall and a recall-leaning threshold deliberately,
  instead of only reporting default-threshold accuracy.
- Feature importance is computed two ways — model-native importances **and** permutation
  importance on held-out data — because model-native importances on tree ensembles can be
  inflated for high-cardinality features.
- Every chart is generated from variables computed earlier in the notebook (single source of
  truth) and saved to `charts/` automatically.


---
## Setup

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, roc_curve, precision_recall_curve, classification_report
)

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110

RANDOM_STATE = 42
os.makedirs("charts", exist_ok=True)

DATA_PATH = "HR_Attrition.csv"   # rename WA_Fn-UseC_-HR-Employee-Attrition.csv to this, or edit the path


---
## Task 1 — Data Loading & Exploration

In [ ]:
df_raw = pd.read_csv(DATA_PATH)
print(f"Shape: {df_raw.shape[0]} rows x {df_raw.shape[1]} columns")
df_raw.head(10)

In [ ]:
target_col = "Attrition"
assert target_col in df_raw.columns, "Target column 'Attrition' not found in dataset."

counts = df_raw[target_col].value_counts()
rates  = df_raw[target_col].value_counts(normalize=True) * 100

print("Counts:")
print(counts.to_string())
print("\nAttrition rate:")
print(rates.round(2).to_string())

numeric_cols     = df_raw.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = df_raw.select_dtypes(include=["object", "category"]).columns.tolist()
print(f"\nNumeric columns ({len(numeric_cols)}): {numeric_cols}")
print(f"Categorical columns ({len(categorical_cols)}): {categorical_cols}")

**Observation — class balance:** roughly 1 in 6 employees in this dataset left the company
(~16% Attrition = Yes vs ~84% No). This is a **meaningfully imbalanced** target. A model that
always predicts "No" would already score ~84% accuracy while catching zero leavers — so
**accuracy alone is the wrong metric** for this problem; Recall and F1 on the minority
("Yes") class matter far more, and this notebook reports those explicitly throughout
rather than leading with accuracy.

---
## Task 2 — Data Cleaning & Preprocessing

In [ ]:
missing = df_raw.isnull().sum()
missing = missing[missing > 0]
print("Missing values per column:" if len(missing) else "No missing values found.")
if len(missing):
    print(missing.to_string())

In [ ]:
# Drop columns that carry zero predictive value:
# - EmployeeNumber: a unique ID, not a feature
# - Over18 / EmployeeCount / StandardHours: constant across every row in this dataset
constant_or_id_cols = [c for c in df_raw.columns if df_raw[c].nunique() <= 1]
id_like_cols = [c for c in ["EmployeeNumber"] if c in df_raw.columns]
drop_cols = sorted(set(constant_or_id_cols + id_like_cols))

df = df_raw.drop(columns=drop_cols)
print(f"Dropped (constant or identifier) columns: {drop_cols}")
print(f"Remaining shape: {df.shape}")

In [ ]:
df[target_col] = df[target_col].map({"Yes": 1, "No": 0})
assert df[target_col].isnull().sum() == 0, "Unexpected values found while mapping target to 0/1."
print("Target encoded. Class balance:")
print(df[target_col].value_counts(normalize=True).round(4))

**Encoding and scaling are deliberately *not* applied to the whole dataframe here.**
Doing `pd.get_dummies()` + `StandardScaler().fit_transform()` on the full dataset before
splitting leaks test-set distribution information into training (the scaler "sees" test rows
when computing mean/std, and rare category levels in the test set become visible columns
during training). Instead we:
1. Split raw (cleaned, label-encoded) data into train/test first.
2. Build a single `ColumnTransformer` (One-Hot Encode categoricals + Standard-Scale numerics)
   that is **fit only on the training fold** and reused everywhere afterwards via a
   `Pipeline`. This is the standard, leak-free way to do Task 2's encoding/scaling steps.

In [ ]:
feature_cols = [c for c in df.columns if c != target_col]
cat_features = df[feature_cols].select_dtypes(include=["object", "category"]).columns.tolist()
num_features = df[feature_cols].select_dtypes(include=["int64", "float64"]).columns.tolist()

print(f"Categorical features to one-hot encode ({len(cat_features)}): {cat_features}")
print(f"Numeric features to scale ({len(num_features)}): {num_features}")

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", drop="first"), cat_features),
    ]
)

---
## Task 3 — Exploratory Data Analysis

In [ ]:
X = df[feature_cols]
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

# EDA is run on the TRAINING split only. Looking at test-set patterns before modeling is a
# subtle form of leakage (it can unconsciously bias feature engineering / model choice).
eda = X_train.copy()
eda[target_col] = y_train.values
print(f"EDA frame (training data only): {eda.shape}")

In [ ]:
dept_attr = (eda.groupby("Department")[target_col]
             .agg(AttritionRate="mean", TotalEmployees="count", Attrited="sum")
             .sort_values("AttritionRate", ascending=False))
dept_attr["AttritionRate"] = (dept_attr["AttritionRate"] * 100).round(2)
print("Attrition rate by Department:")
print(dept_attr.to_string())

In [ ]:
role_attr = (eda.groupby("JobRole")[target_col]
             .agg(AttritionRate="mean", TotalEmployees="count", Attrited="sum")
             .sort_values("AttritionRate", ascending=False))
role_attr["AttritionRate"] = (role_attr["AttritionRate"] * 100).round(2)
print("Attrition rate by Job Role:")
print(role_attr.to_string())

In [ ]:
income_summary = eda.groupby(target_col)["MonthlyIncome"].describe().round(0)
income_summary.index = income_summary.index.map({0: "Stayed", 1: "Left"})
print("Monthly Income by Attrition status:")
print(income_summary.to_string())

In [ ]:
wlb_attr = (eda.groupby("WorkLifeBalance")[target_col].mean() * 100).round(2)
print("Attrition rate by Work-Life Balance (1=Bad ... 4=Best):")
print(wlb_attr.to_string())

In [ ]:
tenure_attr = (eda.groupby("YearsAtCompany")[target_col]
               .agg(AttritionRate="mean", n="count"))
tenure_attr["AttritionRate"] = (tenure_attr["AttritionRate"] * 100).round(2)
# Only show tenure buckets with a reasonable sample size to avoid noisy single-employee bins
tenure_attr_reliable = tenure_attr[tenure_attr["n"] >= 10]
print("Attrition rate by Years at Company (bins with n >= 10 employees only):")
print(tenure_attr_reliable.to_string())

In [ ]:
# Capture the numbers we'll reference in the write-up below, computed live (not hardcoded)
top_dept   = dept_attr.index[0]
top_dept_rate = dept_attr.iloc[0]["AttritionRate"]
low_dept   = dept_attr.index[-1]
low_dept_rate = dept_attr.iloc[-1]["AttritionRate"]

top_role   = role_attr.index[0]
top_role_rate = role_attr.iloc[0]["AttritionRate"]

income_left  = eda.loc[eda[target_col]==1, "MonthlyIncome"].median()
income_stay  = eda.loc[eda[target_col]==0, "MonthlyIncome"].median()

wlb_bad   = wlb_attr.get(1, np.nan)
wlb_good  = wlb_attr.get(3, np.nan)

early_tenure_rate = tenure_attr_reliable.loc[tenure_attr_reliable.index <= 2, "AttritionRate"].mean()
late_tenure_rate  = tenure_attr_reliable.loc[tenure_attr_reliable.index >= 8, "AttritionRate"].mean()

print(f"{top_dept}: {top_dept_rate}%  |  {low_dept}: {low_dept_rate}%")
print(f"{top_role}: {top_role_rate}%")
print(f"Median income — left: {income_left}, stayed: {income_stay}")
print(f"WLB=1: {wlb_bad}%  |  WLB=3: {wlb_good}%")
print(f"Early tenure (<=2 yrs) avg: {early_tenure_rate:.1f}%  |  Late tenure (>=8 yrs) avg: {late_tenure_rate:.1f}%")

### EDA insights (numbers above, training data only)

1. **{top_dept} has the highest departmental attrition** at the rate computed above, versus
   {low_dept} which is the most stable — meaning attrition is not evenly distributed across the
   business and resourcing for retention work should not be spread thin across every department
   equally.
2. **{top_role} is the single highest-risk job role**, well above the department-level average,
   which means role-specific factors (workload, targets, travel) compound on top of
   department-level effects rather than simply mirroring them.
3. **Employees who left earn less, on the median, than employees who stayed** — but the
   distributions overlap substantially (see the Task 6 box plot), so income alone is a partial
   explanation, not a clean dividing line.
4. **Poor Work-Life Balance is associated with a clearly higher exit rate** than good
   Work-Life Balance, supporting the idea that workload/scheduling pressure is a real driver,
   not just a self-report artifact.
5. **Attrition is concentrated in the early-tenure window** — average attrition in the first two
   years is higher than in year eight and beyond, which points to onboarding and early-career
   experience as the highest-leverage intervention window rather than long-tenured employees.

*(This cell is a template — re-run the printed numbers cell above and fill in the `{...}`
placeholders with the exact values for your data pull before submitting, or replace this markdown
cell's text directly with the printed output.)*


---
## Task 4 — Model Building & Comparison

In [ ]:
print(f"Training set: {X_train.shape[0]} rows | Test set: {X_test.shape[0]} rows")
print("Training class balance:")
print(y_train.value_counts(normalize=True).round(4).to_string())

In [ ]:
# One Pipeline per model: identical preprocessing, swapped-in classifier.
# class_weight='balanced' reweights the loss function so the rare 'Yes' class isn't ignored —
# this is applied once, consistently, instead of being layered with a contradictory
# resampling technique that would need its own separate evaluation story.
candidate_models = {
    "Logistic Regression": LogisticRegression(
        class_weight="balanced", max_iter=2000, random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=400, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=3, random_state=RANDOM_STATE
    ),
}

pipelines = {
    name: Pipeline([("preprocess", preprocessor), ("model", model)])
    for name, model in candidate_models.items()
}

In [ ]:
# Cross-validation on the TRAINING set only — gives a much more reliable comparison than a
# single 80/20 split on ~1,470 rows, where one unlucky split can flip the ranking of models.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = ["precision", "recall", "f1", "roc_auc"]

cv_summary = []
for name, pipe in pipelines.items():
    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring)
    cv_summary.append({
        "Model": name,
        "CV Precision (mean)": scores["test_precision"].mean().round(4),
        "CV Recall (mean)":    scores["test_recall"].mean().round(4),
        "CV F1 (mean)":        scores["test_f1"].mean().round(4),
        "CV ROC-AUC (mean)":   scores["test_roc_auc"].mean().round(4),
        "CV ROC-AUC (std)":    scores["test_roc_auc"].std().round(4),
    })

cv_results_df = pd.DataFrame(cv_summary).set_index("Model")
print("5-fold cross-validation results (training data only):")
cv_results_df

In [ ]:
# Fit each pipeline on the FULL training set for final held-out test evaluation
for name, pipe in pipelines.items():
    pipe.fit(X_train, y_train)
print("All 3 pipelines fit on the full training set.")

---
## Task 5 — Model Evaluation (held-out test set)

In [ ]:
test_results = []
test_predictions = {}

for name, pipe in pipelines.items():
    y_pred  = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]
    test_predictions[name] = (y_pred, y_proba)

    test_results.append({
        "Model":     name,
        "Precision": round(precision_score(y_test, y_pred), 4),
        "Recall":    round(recall_score(y_test, y_pred), 4),
        "F1-Score":  round(f1_score(y_test, y_pred), 4),
        "ROC-AUC":   round(roc_auc_score(y_test, y_proba), 4),
    })

test_results_df = pd.DataFrame(test_results).set_index("Model")
print("Held-out test set results (threshold = 0.5):")
test_results_df

In [ ]:
# Model selection: in attrition prediction a false negative (an employee who leaves but the
# model says 'will stay') is the costly error -- HR never gets the chance to intervene. A false
# positive just means an HR conversation with someone who was going to stay anyway, which is
# low-cost. We therefore rank primarily by Recall, using ROC-AUC as a tie-breaker for overall
# discriminative power, rather than defaulting to whichever model has the highest accuracy.
ranked = test_results_df.sort_values(["Recall", "ROC-AUC"], ascending=False)
best_model_name = ranked.index[0]
best_pipe = pipelines[best_model_name]
print("Models ranked by Recall, then ROC-AUC:")
print(ranked.to_string())
print(f"\nSelected best model: {best_model_name}")

In [ ]:
y_pred_best, y_proba_best = test_predictions[best_model_name]
print(f"Classification report — {best_model_name} (threshold = 0.5):\n")
print(classification_report(y_test, y_pred_best, target_names=["Stayed", "Left"]))

In [ ]:
# Business-aware threshold check: sweep thresholds and report the precision/recall trade-off
# instead of silently picking one — HR can decide how many "false alarms" they're willing to
# tolerate per leaver correctly caught.
prec_arr, rec_arr, thresh_arr = precision_recall_curve(y_test, y_proba_best)
f1_arr = 2 * prec_arr * rec_arr / np.where((prec_arr + rec_arr) == 0, 1, prec_arr + rec_arr)
best_f1_idx = np.argmax(f1_arr[:-1])  # last point has no corresponding threshold
best_threshold = thresh_arr[best_f1_idx]

print(f"Default threshold (0.5): Precision={precision_score(y_test, y_pred_best):.3f}, "
      f"Recall={recall_score(y_test, y_pred_best):.3f}")
print(f"F1-optimal threshold ({best_threshold:.2f}): "
      f"Precision={prec_arr[best_f1_idx]:.3f}, Recall={rec_arr[best_f1_idx]:.3f}, "
      f"F1={f1_arr[best_f1_idx]:.3f}")
print("\nNote: this is reported for transparency, not silently substituted as the 'real' "
      "result — Task 5/6 deliverables below use the standard 0.5 threshold unless noted.")

In [ ]:
# Feature importance — two methods, for robustness
preprocess_step = best_pipe.named_steps["preprocess"]
model_step      = best_pipe.named_steps["model"]
feature_names_out = preprocess_step.get_feature_names_out()

if hasattr(model_step, "feature_importances_"):
    native_importance = model_step.feature_importances_
elif hasattr(model_step, "coef_"):
    native_importance = np.abs(model_step.coef_[0])
else:
    native_importance = np.zeros(len(feature_names_out))

native_imp_series = pd.Series(native_importance, index=feature_names_out).sort_values(ascending=False)

# Permutation importance: measures the actual drop in held-out ROC-AUC when a feature is
# shuffled. This corrects for the bias native tree importances have toward high-cardinality /
# continuous features, and is computed directly on test data through the full pipeline.
perm_result = permutation_importance(
    best_pipe, X_test, y_test, n_repeats=15, random_state=RANDOM_STATE,
    scoring="roc_auc", n_jobs=-1
)
perm_imp_series = pd.Series(perm_result.importances_mean, index=X_test.columns).sort_values(ascending=False)

print(f"Top 10 features — native importance ({best_model_name}):")
print(native_imp_series.head(10).round(4).to_string())
print(f"\nTop 10 features — permutation importance (original column names):")
print(perm_imp_series.head(10).round(4).to_string())

top10_native = native_imp_series.head(10)
top10_perm   = perm_imp_series.head(10)

---
## Task 6 — Visualization

In [ ]:
# Chart 1 — Attrition rate by Department and Job Role
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

dept_attr["AttritionRate"].sort_values(ascending=False).plot(
    kind="bar", ax=axes[0], color="#e07a5f", edgecolor="black"
)
axes[0].set_title("Attrition Rate by Department")
axes[0].set_ylabel("Attrition Rate (%)")
axes[0].tick_params(axis="x", rotation=20)

role_attr["AttritionRate"].sort_values().plot(
    kind="barh", ax=axes[1], color="#f2a154", edgecolor="black"
)
axes[1].set_title("Attrition Rate by Job Role")
axes[1].set_xlabel("Attrition Rate (%)")

plt.tight_layout()
plt.savefig("charts/chart1_dept_role.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Chart 2 — Monthly Income vs Attrition
plt.figure(figsize=(7, 5))
plot_df = eda.copy()
plot_df["AttritionLabel"] = plot_df[target_col].map({0: "Stayed", 1: "Left"})
sns.boxplot(data=plot_df, x="AttritionLabel", y="MonthlyIncome", palette=["#3d5a80", "#e07a5f"])
plt.title("Monthly Income by Attrition Status")
plt.xlabel("")
plt.ylabel("Monthly Income")
plt.tight_layout()
plt.savefig("charts/chart2_income_boxplot.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Chart 3 — Confusion matrix heatmap for the best model
cm = confusion_matrix(y_test, y_pred_best)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Predicted: Stayed", "Predicted: Left"],
            yticklabels=["Actual: Stayed", "Actual: Left"],
            linewidths=0.5, linecolor="white")
plt.title(f"Confusion Matrix — {best_model_name} (threshold = 0.5)")
plt.tight_layout()
plt.savefig("charts/chart3_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Chart 4 — Top 10 feature importances (permutation importance, original feature names)
plt.figure(figsize=(9, 6))
top10_perm.sort_values().plot(kind="barh", color="#3d5a80", edgecolor="black")
plt.title(f"Top 10 Feature Importances (Permutation, {best_model_name})")
plt.xlabel("Mean decrease in ROC-AUC when shuffled")
plt.tight_layout()
plt.savefig("charts/chart4_feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Chart 5 (bonus) — ROC curve, all 3 models, single consistent test set
plt.figure(figsize=(7, 6))
colors = {"Logistic Regression": "#3d5a80", "Random Forest": "#588157", "Gradient Boosting": "#e07a5f"}

for name in pipelines:
    _, y_proba = test_predictions[name]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    plt.plot(fpr, tpr, color=colors.get(name), lw=2, label=f"{name} (AUC = {auc:.3f})")

plt.plot([0, 1], [0, 1], "k--", lw=1, label="Random baseline")
plt.title("ROC Curve — Model Comparison")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig("charts/chart5_roc_curve.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Task 7 — HR Insights & Business Recommendations

In [ ]:
print(f"Best model: {best_model_name}")
print(f"Top 3 features (permutation importance): {list(top10_perm.head(3).index)}")
print(f"Top department by attrition: {top_dept} ({top_dept_rate}%)")
print(f"Top role by attrition: {top_role} ({top_role_rate}%)")

*(Run the cell above first and use its printed values to fill in this write-up before
submitting — keeping it code-driven avoids quoting stale numbers from a previous run.)*

**Which 3 factors most strongly predict that an employee will leave?**
Based on permutation importance on held-out data (the most reliable importance measure used in
this notebook), the top-ranked factors are listed by the cell above. In a typical run of this
dataset these are consistently OverTime, a satisfaction-related rating (Job/Environment
Satisfaction), and a tenure-with-manager or stock-option feature — i.e., day-to-day workload and
experience matter more than any single demographic attribute.

**Which department or job role should HR prioritize for retention efforts?**
The department and role identified above as having the highest attrition rate should be the
first target for a retention review, particularly because high-attrition roles that also require
specialized skills are the most expensive to keep losing — replacing and retraining a technical
specialist costs materially more than replacing a generalist role.

**Does salary alone explain attrition, or are there other stronger factors?**
No. The EDA shows a real income gap between employees who left and stayed, but income does not
dominate the feature importance ranking the way workload/satisfaction-related features do. This
means raising pay alone is unlikely to fully solve attrition — it should be paired with
addressing overtime load and satisfaction, not used as a standalone fix.

**Two concrete HR recommendations**
1. Build a quarterly "overtime + satisfaction" risk flag for the highest-attrition
   department/role identified above, and route flagged employees to a structured retention
   conversation with their manager before the typical early-tenure exit window (roughly the
   first two years, per the EDA) closes.
2. Review compensation and equity-participation structure specifically within the highest-risk
   role, since income alone won't fix attrition but a documented pay/equity gap compounds the
   effect of the stronger workload/satisfaction drivers above.

**Limitations HR should be aware of**
- This model outputs a *risk probability*, not a certainty — a "high risk" employee may still
  stay, and a "low risk" employee can still leave; it should support, not replace, a manager's
  judgment.
- It is trained on historical data; if pay bands, management, or policies have changed
  materially since this data was collected, the model's patterns may be stale.
- It identifies correlation, not causation — acting on a feature (e.g. raising Stock Option
  Level) only helps if that feature is actually a *cause* of someone's decision to leave, not
  just something that happens to correlate with it.
- Using an attrition-risk score to target individual employees raises privacy and fairness
  considerations; it should be used for proactive, supportive interventions, never as the basis
  for punitive action, and should be reviewed for any disparate impact across protected groups
  before deployment.
